In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/hgultekin/bbcnewsarchive/bbc-news-data.csv


In [22]:
import pandas as pd
import csv

In [43]:
df = pd.read_csv(
    "/kaggle/input/datasets/hgultekin/bbcnewsarchive/bbc-news-data.csv",
    sep='\t'
)

In [44]:
df.head(2)

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...


In [45]:
df.shape

(2225, 4)

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  2225 non-null   object
 1   filename  2225 non-null   object
 2   title     2225 non-null   object
 3   content   2225 non-null   object
dtypes: object(4)
memory usage: 69.7+ KB


In [47]:
data=df[['content']]

In [48]:
data.head()

,content
0,Quarterly profits at US media giant TimeWarne...
1,The dollar has hit its highest level against ...
2,The owners of embattled Russian oil giant Yuk...
3,British Airways has blamed high fuel prices f...
4,Shares in UK drinks and food firm Allied Dome...


In [49]:
df.isna().sum()

category    0
filename    0
title       0
content     0
dtype: int64

In [50]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('wordnet')
nltk.download('stopwords')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [51]:
def clean_news(text):
    text=re.sub('[^a-zA-Z]'," ",str(text)).lower()
    words=text.split()
    words=[lemmatizer.lemmatize(w) for w in words if w not in stopwords.words('english')]
    return " ".join(words)

In [52]:
data["cleaned"]=data['content'].apply(clean_news)

/tmp/ipykernel_55/2620627314.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["cleaned"]=data['content'].apply(clean_news)


In [54]:
data.head()

,content,cleaned
0,Quarterly profits at US media giant TimeWarne...,quarterly profit u medium giant timewarner jum...
1,The dollar has hit its highest level against ...,dollar hit highest level euro almost three mon...
2,The owners of embattled Russian oil giant Yuk...,owner embattled russian oil giant yukos ask bu...
3,British Airways has blamed high fuel prices f...,british airway blamed high fuel price drop pro...
4,Shares in UK drinks and food firm Allied Dome...,share uk drink food firm allied domecq risen s...


In [55]:
from sklearn.feature_extraction.text import CountVectorizer

# max_df=0.95: Ignore words that appear in more than 95% of documents (too common)
# min_df=2: Word must appear in at least 2 documents
cv = CountVectorizer(max_df=0.95, min_df=2)

dtm = cv.fit_transform(data['cleaned'])

In [56]:
from sklearn.decomposition import LatentDirichletAllocation

In [57]:
lda=LatentDirichletAllocation(n_components=5,random_state=42)
lda.fit(dtm)

LatentDirichletAllocation(n_components=5, random_state=42)

In [58]:
for index, topic in enumerate(lda.components_):
    print(f"THE TOP 15 WORDS FOR TOPIC #{index}")
    print([cv.get_feature_names_out()[i] for i in topic.argsort()[-15:]])
    print('\n')

THE TOP 15 WORDS FOR TOPIC #0
['new', 'price', 'bank', 'firm', 'last', 'sale', 'also', 'market', 'award', 'company', 'film', 'best', 'bn', 'year', 'said']


THE TOP 15 WORDS FOR TOPIC #1
['firm', 'would', 'new', 'mobile', 'also', 'computer', 'phone', 'one', 'user', 'service', 'music', 'game', 'technology', 'people', 'said']


THE TOP 15 WORDS FOR TOPIC #2
['first', 'two', 'could', 'uk', 'mr', 'tv', 'film', 'one', 'people', 'would', 'new', 'show', 'also', 'year', 'said']


THE TOP 15 WORDS FOR TOPIC #3
['plan', 'tax', 'also', 'tory', 'blair', 'say', 'people', 'minister', 'election', 'party', 'labour', 'government', 'would', 'mr', 'said']


THE TOP 15 WORDS FOR TOPIC #4
['team', 'world', 'one', 'last', 'club', 'two', 'back', 'england', 'win', 'player', 'time', 'first', 'year', 'game', 'said']




In [68]:
topic_mapping = {
    0: "Business & Economy",
    1: "Technology",
    2: "Entertainment",
    3: "Politics",
    4: "Sports"
}

In [73]:
def predict_news_topic(text):
    cleaned_text=clean_news(text)
    vec_text=cv.transform([cleaned_text])
    topic_prob=lda.transform(vec_text)
    best_topic_prob=topic_prob.argmax()
    return topic_mapping[best_topic_prob]
    

In [74]:
print(predict_news_topic("The game ended in a draw after the striker missed the goal.")) 
# Output should be 'Sports'

print(predict_news_topic("The prime minister announced new tax cuts ahead of the election.")) 
# Output should be 'Politics'

Sports
Politics


In [75]:
print(predict_news_topic("The peaceful ocean waves crashed gently against the golden sand under the moonlight."))

Business & Economy
